# Lesson 5 Assignment

In this lab assignment, you will build a non-tree-based classifier where you can ensemble any base-learners and pass it to the using a BaggingClassifier or any of the other ensemble learners in sklearn.ensemble.

In [1]:
# import packages
%matplotlib inline
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import seaborn as sns
import pandas as pd
from sklearn.datasets import make_moons
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GridSearchCV
from sklearn.utils import resample

# make this notebook's output stable across runs
np.random.seed(0)

## Data Set Information:

This dataset represents a set of possible advertisements on Internet pages. The features encode the geometry of the image (if available) as well as phrases occuring in the URL, the image's URL and alt text, the anchor text, and words occuring near the anchor text. The task is to predict whether an image is an advertisement ("ad") or not ("nonad"). Additional information can be found [here](https://archive.ics.uci.edu/ml/datasets/internet%2Badvertisements).

## Attribute Information:

The dataset has 3 continous (height, width, aratio) and 1555 binary (urls, tags, captions) features. 

## Source:

Creator & donor: Nicholas Kushmerick <nick '@' ucd.ie>

In [2]:
# Load the data
internetAd = pd.read_csv('Internet_Ad_Data.csv', sep=',', error_bad_lines=False)
print(internetAd.info())
internetAd.head(20)

/tmp/ipykernel_22894/3747926279.py:2: FutureWarning: The error_bad_lines argument has been deprecated and will be removed in a future version. Use on_bad_lines in the future.


  internetAd = pd.read_csv('Internet_Ad_Data.csv', sep=',', error_bad_lines=False)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3279 entries, 0 to 3278
Columns: 1559 entries, height to Target
dtypes: int64(1554), object(5)
memory usage: 39.0+ MB
None


/tmp/ipykernel_22894/3747926279.py:2: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  internetAd = pd.read_csv('Internet_Ad_Data.csv', sep=',', error_bad_lines=False)


,height,width,aratio,local,url*images+buttons,url*likesbooks.com,url*www.slake.com,url*hydrogeologist,url*oso,url*media,...,caption*home,caption*my,caption*your,caption*in,caption*bytes,caption*here,caption*click,caption*for,caption*you,Target
0,125,125,1.0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,ad.
1,57,468,8.2105,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,ad.
2,33,230,6.9696,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,ad.
3,60,468,7.8,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,ad.
4,60,468,7.8,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,ad.
5,60,468,7.8,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,ad.
6,59,460,7.7966,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,ad.
7,60,234,3.9,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,ad.
8,60,468,7.8,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,ad.
9,60,468,7.8,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,ad.


## Question 1: Prepare and impute missing values with the median

In [3]:
obj_feats = ["height","width","aratio","local","Target"] #list of features that have dtype object
for feature in internetAd.columns: internetAd[feature] = internetAd[feature].replace(regex="\?$",value=np.NAN)        #Removes _? and replaces it with NAN
internetAd["Target"] = internetAd["Target"].replace(to_replace={"nonad.","ad."}, value={0,1})                         #Encodes the Target column
for feature in obj_feats: internetAd[feature] = internetAd[feature].astype(float)                                     #Changes dtype of columns that contain objects to columns that contain floats
for feature in internetAd.columns: internetAd[feature] = internetAd[feature].fillna(value=internetAd.mean()[feature]) #Fills in any missing values with the average of that feature

## Question 2: Split dataset into training and test set

In [4]:
from sklearn.model_selection import train_test_split

X = internetAd.drop(columns="Target")
y = internetAd["Target"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

## Question 3: Train and evaluate a LogisticRegression classifier using LogisticRegression.

In [5]:
clf = LogisticRegression(max_iter=800).fit(X_train,y_train) #max_iter < 800 leads to divergences

In [6]:
test_z      = clf.predict(X_test)
test_z_prob = clf.predict_proba(X_test)

y_true = y_test
acc = accuracy_score(y_true, test_z)
auc = roc_auc_score(y_true, test_z)
print("Accuracy: {}%. \n ROC AUC: {}.".format(round(acc*100,2), round(auc,2)))

Accuracy: 96.31%. 
 ROC AUC: 0.89.


## Question 4: Use BaggingClassifier to train and evaluate an ensemble model of LogisticRegression  base classifiers. Each base classifier should be trained only on a sample half the size of the training data, and using only half as many features as there are in in total the training data (read the documentation for the function to see how to do this).

In [7]:
max_samples, max_features = int(len(X_train)/2), int(len(internetAd.columns)/2)
bagOLR = BaggingClassifier(max_samples=max_samples,max_features=max_features).fit(X_train,y_train)

In [8]:
test_z      = bagOLR.predict(X_test)
test_z_prob = bagOLR.predict_proba(X_test)

y_true = y_test
acc = accuracy_score(y_true, test_z)
auc = roc_auc_score(y_true, test_z)
print("Accuracy: {}%. \n ROC AUC: {}.".format(round(acc*100,2), round(auc,2)))

Accuracy: 96.4%. 
 ROC AUC: 0.91.


## Question 5: Use AdaBoostClassifier to train and evaluate an ensemble model of LogisticRegression base classifiers.

In [9]:
boostOkLR = AdaBoostClassifier().fit(X_train,y_train)

In [10]:
test_z      = boostOkLR.predict(X_test)
test_z_prob = boostOkLR.predict_proba(X_test)

y_true = y_test
acc = accuracy_score(y_true, test_z)
auc = roc_auc_score(y_true, test_z)
print("Accuracy: {}%. \n ROC AUC: {}.".format(round(acc*100,2), round(auc,2)))

Accuracy: 95.94%. 
 ROC AUC: 0.89.


## Question 6. Create a new text cell in your Notebook: Complete a 50-100 word summary (or short description of your thinking in applying this week's learning to the solution) of your experience in this assignment. Include:
                                                                      
* What was your incoming experience with this model, if any? 
* What steps you took, what obstacles you encountered.
* How you link this exercise to real-world, machine learning problem-solving. (What steps were missing? What else do you need to learn?) 
> This summary allows your instructor to know how you are doing and allot points for your effort in thinking and planning, and making connections to real-world work.